In [5]:
import sys
import os
import io
# 添加项目根目录到 Python 路径
from utils import model_untils


# =============================================================================
# LangGraph 入门课件
# =============================================================================
#
# 用途：教学演示 - 使用 LangGraph 构建 AI 智能体工作流
#
# 核心概念：
#   - StateGraph = "有状态的工作流图"
#   - Node = 图中的处理节点（函数）
#   - Edge = 节点之间的连接边
#   - State = 贯穿整个图的状态数据
#   - 支持循环、条件分支等复杂流程
# =============================================================================

# -----------------------------------------------------------------------------
# 运行前检查
# -----------------------------------------------------------------------------
# 1. 已安装依赖：pip install langgraph langchain-core langchain-openai
# 2. 已配置 .env 文件中的 ALIYUN_API_KEY
# -----------------------------------------------------------------------------


# =============================================================================
# 第一部分：理解 LangGraph 核心概念
# =============================================================================
"""
什么是 LangGraph？

📊 定义
   LangGraph = 基于图结构的状态机框架
   用于构建有状态的多参与者（Actor）AI 应用

🎯 为什么需要 LangGraph？

   普通的 LangChain 链（Chain）是线性的：
   A → B → C（单向、无循环）

   但真实的 AI 应用往往是循环的：
   - Agent 思考 → 调用工具 → 观察结果 → 再思考 → 再调用 → ...
   - 需要记忆历史状态
   - 需要根据条件走不同分支

   LangGraph 就能解决：
   ✅ 循环工作流（Agent 反复思考-行动）
   ✅ 条件分支（根据结果决定下一步）
   ✅ 状态管理（全程共享数据）
   ✅ 可视化调试（生成流程图）

💡 生活化比喻
   LangGraph = "地铁线路图"
   - 站点（Node）= 每个处理步骤
   - 线路（Edge）= 站点间的连接
   - 乘客（State）= 在图中传递的数据
   - 换乘站（Conditional Edge）= 根据条件选择不同线路
"""


# =============================================================================
# 示例 1: 最简图 - 两个节点的顺序执行
# =============================================================================

def simple_two_node_graph():
    """
    最简单的 LangGraph：两个节点顺序执行

    START → 问候 → 回应 → END
    """
    print("=" * 60)
    print("示例 1: 最简图 - 两个节点顺序执行")
    print("=" * 60)

    from typing import TypedDict
    from langgraph.graph import StateGraph, START, END

    # 1. 定义状态：图中传递的数据结构
    #    TypedDict 是 Python 的类型提示工具，LangGraph 用它来定义"贯穿整张图的状态"
    #    每个节点函数都可以读取和修改这个状态
    class GraphState(TypedDict):
        """图的状态：存储消息和步骤计数"""
        messages: list[str]
        step_count: int

    # 2. 定义节点函数：每个节点接收状态，返回状态更新
    #    ⚠️ 关键：节点函数的返回值不是"替换"整个状态，而是"合并"到状态中
    #    例如 return {"messages": [...]} 只会更新 messages 字段，step_count 不受影响
    def greet(state: GraphState):
        """节点 1：添加问候消息"""
        print("  [节点: greet] 添加问候语")
        return {
            "messages": ["你好！很高兴见到你！"],
            "step_count": 1
        }

    def respond(state: GraphState):
        """节点 2：添加回应消息
        这里演示了如何读取之前的状态（state["step_count"]）并在此基础上更新
        """
        print("  [节点: respond] 添加回应")
        return {
            "messages": ["你好！我也很高兴！"],
            "step_count": state["step_count"] + 1  # 读取上一步的值，+1
        }

    # 3. 构建图：连接节点
    #    StateGraph(GraphState)  — 创建一个带状态的图，状态类型是 GraphState
    #    .add_node("名字", 函数)  — 注册一个节点，"名字" 用于后续连线时引用
    #    .add_edge(A, B)         — 添加一条从 A 到 B 的固定边（无条件，一定执行）
    #    START                   — 图的入口常量，等价于"从这里开始"
    #    END                     — 图的出口常量，等价到这里结束
    #    .compile()              — 把图编译成可执行的对象
    graph = (
        StateGraph(GraphState)
        .add_node("greet", greet)       # 添加节点
        .add_node("respond", respond)
        .add_edge(START, "greet")       # 起点 → greet
        .add_edge("greet", "respond")   # greet → respond
        .add_edge("respond", END)       # respond → 终点
        .compile()
    )



   #
   # #    返回值是最终的状态（所有节点执行完毕后的状态）
    print("【执行图】")
    result = graph.invoke({"messages": [], "step_count": 0})
    print(f"  最终消息: {result['messages']}")
    print(f"  总步骤数: {result['step_count']}")
    print()

    from IPython.display import Image, display
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))